In [67]:
import pandas as pd
import numpy as np

# --- 1. Load Files ---
# Ensure these files are in the same directory as your script
excel_file = '../Data/Funds.xlsx'
master_df = pd.read_excel(excel_file, sheet_name='Master')
sheet1_df = pd.read_excel(excel_file, sheet_name='Sheet1')

# Keep reading the historical file as CSV
history_df = pd.read_csv('../Data/portfolios_cleaned.csv')

In [68]:

# --- 2. Fix Master Headers ---
# The Morningstar macro places 'Fund ID' in A1, but the rest of the headers are shifted down on row 2.
headers = master_df.iloc[0].tolist()
headers[0] = 'Fund ID' 
master_df.columns = headers
master_df = master_df.drop(0).reset_index(drop=True)

# Drop any trailing blank rows
master_df = master_df.dropna(subset=['Fund ID'])


In [69]:
sheet1_df

,Norwaz Equity Active Funds - ISIN,Unnamed: 1,Unnamed: 2,Norwaz Equity Active Funds - SECID
0,NO0010089576,ABIF Norge ++ Acc,F0GBR04NEN,F0GBR04NEN
1,NO0010105497,Alfred Berg Aktiv II,F0GBR04NHC,F0GBR04NHC
2,NO0010089444,Alfred Berg Aktiv R (NOK),F0GBR04NE5,F0GBR04NE5
3,NO0010105489,Alfred Berg Gambak R (NOK),F0GBR04NHA,F0GBR04NHA
4,NO0010032055,Alfred Berg Humanfond R (NOK),F0GBR04P1G,F0GBR04P1G
...,...,...,...,...
112,NO0010080815,NaN,NaN,F0GBR04HKF
113,NO0008000841,NaN,NaN,F0GBR04HH6
114,NO0008000999,NaN,NaN,F0GBR04OUW
115,NO0008001849,NaN,NaN,F0GBR04OX5


In [70]:

# --- 3. Build the Mapping Dictionary ---
# In Sheet1, 'Unnamed: 1' contains the Fund Name, and 'Unnamed: 2' contains the SECID
secid_to_name = dict(zip(sheet1_df['Unnamed: 1'], sheet1_df['Norwaz Equity Active Funds - SECID']))


In [71]:

# --- 4. Reshape (Melt) the 2024 Data ---
# Dynamically grab all columns that represent dates (they start with '2024')
date_columns = [col for col in master_df.columns if str(col).startswith('2024')]

melted_df = master_df.melt(
    id_vars=['Fund ID', 'ISIN', 'Name'],
    value_vars=date_columns,
    var_name='date',
    value_name='weight'
)


In [72]:

# --- 5. Clean the Melted Data ---
# Convert weights to numeric (this safely turns '-N/A' and 'Processing...' strings into NaNs)
melted_df['weight'] = pd.to_numeric(melted_df['weight'], errors='coerce')

# Drop invalid, missing, or zero weights
melted_df = melted_df.dropna(subset=['weight'])
melted_df = melted_df[melted_df['weight'] > 0]

# Standardize date format to YYYY-MM-DD to match historical data
melted_df['date'] = pd.to_datetime(melted_df['date']).dt.strftime('%Y-%m-%d')


In [73]:

# --- 6. Map Names and Format Columns ---
# Attach the Fund Name using our SECID dictionary
melted_df['fund_name'] = melted_df['Fund ID'].map(secid_to_name)

# If any mapping fails (e.g., a fund was missing from Sheet 1), use the Fund ID as a fallback name
melted_df['fund_name'] = melted_df['fund_name'].fillna(melted_df['Fund ID'])

# Rename the 2024 columns to perfectly match the historical dataset
melted_df = melted_df.rename(columns={
    'Fund ID': 'fund_sec_id',
    'ISIN': 'stock_isin',
    'Name': 'stock_name'
})

# Isolate only the columns needed for the ML pipeline
cols_to_keep = ['fund_name', 'fund_sec_id', 'date', 'stock_name', 'stock_isin', 'weight']
clean_2024_df = melted_df[cols_to_keep].copy()

# Drop the missing values in stock_isin columns (non-stock entries - e.g., cash, bonds, etc.)
clean_2024_df = clean_2024_df.dropna(subset=['stock_isin'])

# 2. Force all dates to month-end BEFORE saving to CSV
clean_2024_df['date'] = pd.to_datetime(clean_2024_df['date']) + pd.offsets.MonthEnd(0)


In [74]:

# --- 7. Concatenate with Historical Data ---
print(f"Historical Data Shape: {history_df.shape}")
print(f"Cleaned 2024 Data Shape: {clean_2024_df.shape}")

# Stack the two dataframes together
master_combined = pd.concat([history_df, clean_2024_df], ignore_index=True)

# Sort strictly by fund name, date (chronological), and holding weight (descending)
master_combined['date'] = pd.to_datetime(master_combined['date'])
master_combined = master_combined.sort_values(
    by=['fund_name', 'date', 'weight'], 
    ascending=[True, True, False]
)

# Re-apply the string formatting for the date column
master_combined['date'] = master_combined['date'].dt.strftime('%Y-%m-%d')


Historical Data Shape: (315123, 14)
Cleaned 2024 Data Shape: (27969, 6)


In [75]:

# --- 8. Save the Final File ---
output_filename = '../Data/portfolios_master_updated.csv'
master_combined.to_csv(output_filename, index=False)

print(f"Final Combined Shape: {master_combined.shape}")
print(f"Successfully saved to {output_filename}")

Final Combined Shape: (343092, 14)
Successfully saved to ../Data/portfolios_master_updated.csv
